In [245]:
import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


kaggle里如果要安装东西，就!pip install ... !代表交给系统命令执行 

In [246]:
from xgboost import XGBRegressor

In [247]:
train = pd.read_csv(
    '/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv'
)

test = pd.read_csv(
    '/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv'
)

In [248]:
print(train.shape)
print(test.shape)

train.columns

(1460, 81)
(1459, 80)


Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

In [249]:
X = train.drop(columns = 'SalePrice')
y = train['SalePrice']

print(X.shape)
print(y.shape)

(1460, 80)
(1460,)


In [250]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
)

print(X_train.shape)
print(X_valid.shape)
print(y_train.shape)
print(y_valid.shape)

(1168, 80)
(292, 80)
(1168,)
(292,)


In [251]:
categorical_cols = X_train.select_dtypes(include=['object']).columns
numerical_cols = X_train.select_dtypes(exclude=['object']).columns

print("Categorical columns:", len(categorical_cols))
print("Numerical columns:", len(numerical_cols))

Categorical columns: 43
Numerical columns: 37


In [252]:
numerical_transformer = SimpleImputer(
    strategy = 'median',
)

In [253]:
categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)

In [254]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

In [255]:
y_train_log = np.log1p(y_train)

In [256]:
xgb = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=1,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0,
    reg_lambda=1,

    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)

In [257]:
model = Pipeline(
        steps = [("preprocessor", preprocessor),
                ("regressor", xgb),
                ]
)

In [258]:
model.fit(
    X_train,
    y_train_log,
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.03,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=3, max_leaves=None, min_child_weight=1,
                              missing=nan, monotone_constraints=None,
                              multi_strategy=None, n_estimators=1000, n_jobs=-1,
                              num_parallel_tree=None, ...))])

In [259]:
pred = model.predict(X_valid)

score = np.sqrt(
    mean_squared_error(
        np.log1p(y_valid),
        pred,
    )
)

print("Without log(target):", 0.1535781039872966)
print("With log(target):", 0.14637433145697762)
print("gbr_5cvscores:", 0.12989884231993495)
print(f"gbr: {score}")

Without log(target): 0.1535781039872966
With log(target): 0.14637433145697762
gbr_5cvscores: 0.12989884231993495
gbr: 0.1281656293404639


**下面这里是要看一下不同的训练测试集计算出的score**

In [260]:
kf = KFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42,
)

cv_scores = abs(cross_val_score(
    model,
    X,
    np.log1p(y),
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
))

print(cv_scores)
print(cv_scores.mean())
print(cv_scores.std())

[0.1291721  0.11184019 0.16397011 0.12591087 0.10544337]
0.12726732808062952
0.020331474683795897


In [262]:
grid_gamma = GridSearchCV(
    estimator=model,
    param_grid={
        'regressor__gamma': [0, 0.01, 0.05, 0.1, 0.2, 0.5]
    },
    scoring='neg_root_mean_squared_error',
    cv=kf,
    n_jobs=-1,
    verbose=1
)

grid_gamma.fit(X_train, y_train_log)

print(grid_gamma.best_params_)
print(-grid_gamma.best_score_)

Fitting 5 folds for each of 6 candidates, totalling 30 fits
{'regressor__gamma': 0}
0.1229800089645662
